# Analysis code for final benchmarks

Python imports used (maybe not all are required for this notebook):
* `jupyter`
* `pandas`
* `matplotlib`
* `altair`
* `vega_datasets`
* `altair_transform`
* `altair_data_server`
* `altair_saver`

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import altair as alt
import json

import altair as alt
import vegafusion as vf
vf.enable(
    mimetype="html",  # switch to "png" to make plots show offline in notebook (but they will be blurry for reasons I don't understand)
    embed_options={"scaleFactor": 4},  # scaleFactor 4 makes exported png look good
    row_limit=30000,  # allows you to include plots with more datapoints
)



## Get the data

Data is stored in repo `benchmark-data` and under git commit `2890bcb2`.  Check that this is correct:

In [ ]:
# data is stored locally
benchmark_data_repo = Path("data/benchmark-data/")
benchmark_data_repo_copy = Path("data/benchmark-data-copy/")
benchmark_data_repo_copy2 =  Path("data/benchmark-data-copy2/")
expected_benchmark_data_commit = "2245e0f2e93f01ba9d81b0204801417ce4cd7ce0"
expected_benchmark_data_copy_commit = "2890bcb2"
#expected_benchmark_data_copy2_commit = "cc9b79d90de10754941272f46c960780acb13ad8"
expected_benchmark_data_copy2_commit = "747cbb659a1de5df45735dd5c54072efedee7593"

In [ ]:
display(f"Expected commit: {expected_benchmark_data_commit}")
! cd {benchmark_data_repo} ; git status

In [ ]:
display(f"Expected commit: {expected_benchmark_data_copy_commit}")
! cd {benchmark_data_repo_copy} ; git status

In [ ]:
display(f"Expected commit: {expected_benchmark_data_copy2_commit}")
! cd {benchmark_data_repo_copy2} ; git status

In [ ]:
#commits = [
#    ("GNN No Definitions Update no definitions", "31340a3ba5e8db2f6dbd2255d31a95051b2374e6"),
#    ("GNN Names Update no definitions", "9f01da026f7561d131466bed236b0f7929bb8ce8"),
#    ("GNN Names Update new definitions", "17334d40ff2b6bf9371b55a6aa910fbf419e045c"),
#    ("GNN Names Update all definitions", "8918f8d83757012c77ad06df6905b703e12cb79a"),
#    ("GNN No names Update no definitions", "25a4ec030ea675ec81ab3dd015aa5bdb881f0316"),
#    ("GNN No names Update new definitions", "554a25038e160b0fc7c2524c735b52e64640c5f4"),
#    ("GNN No names Update all definitions", "a930b82adce45b52d5432e28ede3c7827694e649"),
#    ("LSHF", "95e7cd05a47a9a3371f344b55f81b71330d284bb"),
#    ("LSHF extreme tactic decomposition", "8afbb33e2c5c9d12a77dcc06521982f2fcff936b"),
#    ("k-NN", "fd237880a29605f14956a67abed39ca523eb2511"),
#    ("CoqHammer Z3", "213865fd569f30f68ae13a3fef52a2afbc6bbddc"),
#    ("CoqHammer Eprover", "a1d313cae4a144226bda68337eaaf9994c85bfaa"),
#    ("CoqHammer CVC4", "f1f66a2328b005f541522ec8495538be600ec64f"),
#    ("CoqHammer Vampire", "7fc35d4482f8c8a661086ea56bcd0726ec9ad33d"),
#    ("CoqHammer Vampire2", "a8c6f6c1763e07a5c1749956f52eea927932b4d1"),
#    ("CoqHammer 'best' tactic", "c339f3e9398098cd991ddb7d131527ffc9b0ce6e"),
#    ("firstorder eauto with *", "499b10a6f5ed8a2e1f7899af8d6f7413dcdbea45"),
#    ("Transformer GPU", "ef5b7be5c3bcd565378b284602839b002b36a979"),
#    ("Transformer GPU", "5a0d0458fc33c9a1f8aba42a564307b822555956"),
#    ("GNN No names Update new definitions - better validation", "aee2891eb18c9fe881feb99d98d6f65cdab58aa6"),
#    ("Transformer CPU Small", "ef4a03146719d5c4e64e9df95f16775061c5f24b"),
#    ("Transformer CPU Big", "2f876b22fe3e4bbd51b69cf255d6b64e643f9f0e"),
#]
commits = [
    ("GNN No Definitions Update no definitions", "31340a3ba5e8db2f6dbd2255d31a95051b2374e6"),
    ("GNN Names Update no definitions", "9f01da026f7561d131466bed236b0f7929bb8ce8"),
    ("GNN Names Update new definitions", "17334d40ff2b6bf9371b55a6aa910fbf419e045c"),
    ("GNN Names Update all definitions", "8918f8d83757012c77ad06df6905b703e12cb79a"),
    ("GNN No names Update no definitions", "25a4ec030ea675ec81ab3dd015aa5bdb881f0316"),
    ("GNN No names Update new definitions", "554a25038e160b0fc7c2524c735b52e64640c5f4"),
    ("GNN No names Update all definitions", "a930b82adce45b52d5432e28ede3c7827694e649"),
    ("LSHF", "95e7cd05a47a9a3371f344b55f81b71330d284bb"),
    ("LSHF extreme tactic decomposition", "8afbb33e2c5c9d12a77dcc06521982f2fcff936b"),
    ("k-NN", "fd237880a29605f14956a67abed39ca523eb2511"),
    ("Transformer GPU", "ef5b7be5c3bcd565378b284602839b002b36a979"),
    ("Transformer GPU", "5a0d0458fc33c9a1f8aba42a564307b822555956"),
    ("CoqHammer Z3", "a5aba28827b7f32bfabdd8345b7ce56bcfdd9d43"),
    ("CoqHammer Eprover", "03499425f156a2b259cf1e583b705f855103e273"),
    ("CoqHammer CVC4", "d2d606f9239fa87d31bffa606da604fbbe15774b"),
    ("CoqHammer Vampire", "a8c6f6c1763e07a5c1749956f52eea927932b4d1"),
    ("CoqHammer 'best' tactic", "c339f3e9398098cd991ddb7d131527ffc9b0ce6e"),
    ("CoqHammer 'sauto' tactic", "37069316366fe742a0513fd0d45d5f39a40a48ec"),
    ("firstorder eauto with *", "499b10a6f5ed8a2e1f7899af8d6f7413dcdbea45"),
    ("GNN No names Update new definitions - better validation", "aee2891eb18c9fe881feb99d98d6f65cdab58aa6"),
    ("Transformer CPU Small", "ef4a03146719d5c4e64e9df95f16775061c5f24b"),
    ("Transformer CPU Big", "2f876b22fe3e4bbd51b69cf255d6b64e643f9f0e"),
    ("k-NN on CoqGym", "402c79ececfafacb1aeedfb7fcf0959f63aeb164"),
    ("CoqHammer-Z3 on CoqGym", "8a47740efd714777108328c7112c459ee67e40ba"),
    ("CoqHammer-Eprover on CoqGym", "ec67874ecee37c203e3ef4c2e35c91ca1ca08b3f"),
    ("CoqHammer-CVC4 on CoqGym", "ae2c951c394df41e5ef9285c9abd030cd1a8ffb2"),
    ("CoqHammer-Vampire on CoqGym", "d41b24a11fb6cf76b65b1493bd88bde25bc45284"),
    ("CoqHammer-best on CoqGym", "b8ab3ffbc4dba6164e929b9af36f0d70ccea32c5"),
]

In [ ]:
benchmark_paths = [
    {"label": label, "path": f"{commit}/Set-Tactician-Benchmark-{seconds}./combined.bench", "version": "new"}
    for label, commit in commits
    for seconds in [300, 600, 900]
]
benchmark_paths

In [ ]:
def get_df(path: Path):
  df = pd.read_csv(path, sep="\t", names=["package", "theorem", "time_limit", "path", "found_proof", "time", "steps"], dtype=str)
  df["solved"] = df["path"].notna()
  return df

def get_df2(path: Path):
  df = pd.read_csv(path, sep="\t", names=["package", "theorem", "time_limit", "time", "steps", "messages", "path", "found_proof"], dtype=str)
  df["solved"] = df["path"].notna()
  df["error"] = df["time"].isna()
  return df

dfs = []
for d in benchmark_paths:
  if str(d["path"]).startswith("/"):
    path = d["path"]
  else:
    path = benchmark_data_repo / d["path"]
    if not path.exists():
      path = benchmark_data_repo_copy / d["path"]
      if not path.exists():
        path = benchmark_data_repo_copy2 / d["path"]
        if not path.exists():
          continue
  if "version" in d and d["version"] == "new":
    df = get_df2(path)
  else:
    df = get_df(path)
  df["run"] = d["label"]
  dfs.append(df)

results_df = pd.concat(dfs)

# fill in omitted theorems
results_df["omitted"] = False
results_df["package_theorem"] = results_df["package"] + "_" + results_df["theorem"]
results_df["package_theorem"] = pd.Categorical(results_df["package_theorem"], categories=results_df["package_theorem"].unique())
thms = results_df.groupby("package_theorem")["theorem"].first()
pkgs = results_df.groupby("package_theorem")["package"].first()
results_df = results_df.groupby(["run", "package_theorem"], as_index=False).first()
results_df["package_theorem"] = results_df["package_theorem"].astype("str")
results_df["omitted"] = results_df["omitted"].fillna(True)
results_df["error"] = results_df["error"].fillna(True)  # includes omitted rows
results_df["solved"] = results_df["solved"].fillna(False)  # includes omitted rows
results_df["theorem"] = results_df["theorem"].fillna(results_df["package_theorem"].map(thms))
results_df["package"] = results_df["package"].fillna(results_df["package_theorem"].map(pkgs))

results_df["time_limit"] = results_df["time_limit"].astype("float")
results_df["time"] = results_df["time"].astype("float")

thm_lists = {}
for k, df in results_df[results_df["run"] == "k-NN on CoqGym"].groupby("time_limit"):
  thm_lists[k] = list(df["theorem"])

results_df["short_list"] = results_df["theorem"].isin(thm_lists[600.0])
results_df["long_list"] = results_df["theorem"].isin(thm_lists[600.0])  # use same list for long and short

results_2000_df = results_df[results_df["short_list"]].copy()
results_2000_df["solved"] = results_2000_df["solved"] & (results_2000_df["time"] <= 600.0)

results_500_each_df = results_df[results_df["long_list"]].copy()
results_500_each_df["solved"] = results_500_each_df["solved"] & (results_500_each_df["time"] <= 300.0)

del results_df

In [ ]:
results_2000_df

In [ ]:
results_500_each_df

## Make artificial runs with combos

In [ ]:
def get_combo(results_time_df, combo, time_limit):
    mask = np.zeros(len(results_time_df), dtype=bool)
    for run, fraction in combo.items():
        mask = mask | (results_time_df[run] <= time_limit * fraction)
    
    results_time_df = results_time_df[mask].copy()

    scales = {run: 1/np.array(fraction) for run, fraction in combo.items()}
    for run, scale in scales.items():
        results_time_df[run] = results_time_df[run] * scale
    
    results_time_df = results_time_df[list(combo.keys())].reset_index()
    results_time_df = pd.melt(results_time_df, id_vars=["theorem"], var_name='run', value_name='time')
    results_time_df = results_time_df.sort_values("time")
    results_time_df = results_time_df.groupby("theorem").first()
    return results_time_df

def get_combo_size(results_time_df, combo, time_limit):
    mask = np.zeros(len(results_time_df), dtype=bool)
    for run, fraction in combo.items():
        mask = mask | (results_time_df[run] <= time_limit * fraction)
    return mask.sum()

def get_combo_time(results_time_df, combo, time_limit):
    df = get_combo(results_time_df, combo, time_limit)
    return np.minimum(df["time"], 600.0).sum()

def make_combo_results_df(results_df, combo, run_name, time_limit):
    results_time_df = results_df.copy()
    results_time_df["time"] = np.where(results_time_df["solved"], results_time_df["time"], np.inf)
    results_time_df = results_time_df.pivot_table(index="theorem", columns="run", values="time")
    results_time_df = results_time_df.fillna(np.inf)

    df = results_df.copy()
    df = df[df["run"].isin(combo.keys())]
    df = df.drop(columns="time")
    combined_df = get_combo(results_time_df, combo, time_limit)
    combined_df = combined_df.reset_index().set_index(["theorem", "run"])
    combined_df = combined_df.join(df.set_index(["theorem", "run"]))
    # add back in all unsolved theorems
    # just use data from first available run for each theorem
    df = df.groupby("package_theorem").first().reset_index()
    df = df.set_index(["package_theorem", "theorem", "package"])[[]]
    combined_df = df.join(combined_df.reset_index().set_index(["package_theorem", "theorem", "package"])).reset_index()
    combined_df["run"] = run_name
    combined_df["solved"] = combined_df["solved"].fillna(False)
    combined_df["error"] = combined_df["error"].fillna(False)
    combined_df["omitted"] = combined_df["omitted"].fillna(False)
    combined_df["steps"] = combined_df["steps"].fillna(0)
    combined_df["messages"] = combined_df["messages"].fillna(0)
    return combined_df
    
    

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("CoqHammer") & ~df["run"].str.contains("CoqGym")]
coq_hammers = df["run"].unique()
combo = [run for run in coq_hammers if "sauto" not in run]
combo = {run: 1.0/len(combo) for run in combo}
print(combo)
combined_hammer_2000_df = make_combo_results_df(results_2000_df, combo, run_name="CoqHammer combined", time_limit=10.0*60.0)
combined_hammer_2000_df

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("CoqHammer") & df["run"].str.contains("CoqGym")]
coq_hammers = df["run"].unique()
combo = [run for run in coq_hammers if "sauto" not in run]
combo = {run: 1.0/len(combo) for run in combo}
print(combo)
combined_hammer_coqgym_2000_df = make_combo_results_df(results_2000_df, combo, run_name="CoqHammer combined on CoqGym", time_limit=10.0*60.0)
combined_hammer_coqgym_2000_df

In [ ]:
df = results_500_each_df.copy()
df = df[df["run"].str.contains("CoqHammer") & ~df["run"].str.contains("CoqGym")]
coq_hammers = df["run"].unique()
combo = [run for run in coq_hammers if "sauto" not in run]
combo = {run: 1.0/len(combo) for run in combo}
print(combo)
combined_hammer_500_each_df = make_combo_results_df(results_500_each_df, combo, run_name="CoqHammer combined", time_limit=5*60.0)
combined_hammer_500_each_df

In [ ]:
df = results_500_each_df.copy()
df = df[df["run"].str.contains("CoqHammer") & df["run"].str.contains("CoqGym")]
coq_hammers = df["run"].unique()
combo = [run for run in coq_hammers if "sauto" not in run]
combo = {run: 1.0/len(combo) for run in combo}
print(combo)
combined_hammer_coqgym_500_each_df = make_combo_results_df(results_500_each_df, combo, run_name="CoqHammer combined on CoqGym", time_limit=5*60.0)
combined_hammer_coqgym_500_each_df

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("CoqHammer") & ~df["run"].str.contains("CoqGym")]
combo = [run for run in coq_hammers if "sauto" not in run]
combo = {run: 1.0 for run in combo}
print(combo)
combined_hammer_no_rescale_2000_df = make_combo_results_df(
    results_2000_df,
    combo,
    run_name="CoqHammer combined (unscaled)",
    time_limit=10*60.0
)
combined_hammer_no_rescale_2000_df

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("CoqHammer") & df["run"].str.contains("CoqGym")]
combo = [run for run in coq_hammers if "sauto" not in run]
combo = {run: 1.0 for run in combo}
print(combo)
combined_hammer_no_rescale_coqgym_2000_df = make_combo_results_df(
    results_2000_df,
    combo,
    run_name="CoqHammer combined (unscaled) on CoqGym",
    time_limit=10*60.0
)
combined_hammer_no_rescale_coqgym_2000_df

In [ ]:
df = results_500_each_df.copy()
df = df[df["run"].str.contains("CoqHammer") & ~df["run"].str.contains("CoqGym")]
coq_hammers = df["run"].unique()
combo = [run for run in coq_hammers if "sauto" not in run]
combo = {run: 1.0 for run in combo}
print(combo)
combined_hammer_no_rescale_500_each_df = make_combo_results_df(results_500_each_df, combo, run_name="CoqHammer combined (unscaled)", time_limit=5*60.0)
combined_hammer_no_rescale_500_each_df

In [ ]:
df = results_500_each_df.copy()
df = df[df["run"].str.contains("CoqHammer") & df["run"].str.contains("CoqGym")]
coq_hammers = df["run"].unique()
combo = [run for run in coq_hammers if "sauto" not in run]
combo = {run: 1.0 for run in combo}
print(combo)
combined_hammer_no_rescale_coqgym_500_each_df = make_combo_results_df(results_500_each_df, combo, run_name="CoqHammer combined (unscaled) on CoqGym", time_limit=5*60.0)
combined_hammer_no_rescale_coqgym_500_each_df

In [ ]:
df = results_2000_df.copy()
df = df[df["run"].str.contains("CoqHammer") & ~df["run"].str.contains("CoqGym")]
coq_hammers = df["run"].unique()
combo = [run for run in coq_hammers if "Vampire" in run or "best" in run]
combo = {run: 1.0/len(combo) for run in combo}
print(combo)
coq_hammer_vampire_best_2000_df = make_combo_results_df(results_2000_df, combo, run_name="CoqHammer best+vampire", time_limit=10*60.0)
coq_hammer_vampire_best_2000_df

In [ ]:
two_best_models_combined_2000_df = make_combo_results_df(
    results_2000_df, 
    combo= {"k-NN": 1.0/2, "GNN No names Update new definitions": 1.0/2},
    run_name="Top 2 combined",
    time_limit=10*60.0
)
two_best_models_combined_2000_df

In [ ]:
two_best_models_combined_500_each_df = make_combo_results_df(
    results_500_each_df, 
    combo= {"k-NN": 1.0/2, "GNN No names Update new definitions": 1.0/2},
    run_name="Top 2 combined",
    time_limit=5*60.0
)
combined_hammer_500_each_df = make_combo_results_df(results_500_each_df, combo, run_name="CoqHammer combined", time_limit=5*60.0)
two_best_models_combined_500_each_df

In [ ]:
#combo = {"k-NN": 10.0*60.0/3, "GNN No names Update new definitions": 10.0*60.0/3}
#combo.update({ch: t/3 for ch, t in coq_hammers_combo.items()})
#three_best_models_combined_2000_df = make_combo_results_df(
#    results_2000_df, 
#    combo=combo,
#    run_name="Top 3 combined"
#)
#three_best_models_combined_2000_df

In [ ]:
combo = {'GNN No names Update no definitions': 1.0/4, 'k-NN': 1.0/4, 'Transformer GPU': 1.0/4, "CoqHammer 'best' tactic": 1.0/4}
four_best_models_combined_2000_df = make_combo_results_df(
    results_2000_df, 
    combo=combo,
    run_name="Top 4 combined",
    time_limit=10*60.0
)
four_best_models_combined_2000_df


In [ ]:
combo = {run: 10*60.0/len(results_2000_df["run"].unique()) for run in results_2000_df["run"].unique()}
all_combined_2000_df = make_combo_results_df(
    results_2000_df, 
    combo=combo,
    run_name="All combined",
    time_limit=10*60.0
)
all_combined_2000_df

In [ ]:
results_combined_2000_df = pd.concat([
    results_2000_df,
    combined_hammer_2000_df,
    combined_hammer_coqgym_2000_df,
    combined_hammer_no_rescale_2000_df,
    combined_hammer_no_rescale_coqgym_2000_df,
    coq_hammer_vampire_best_2000_df,
    two_best_models_combined_2000_df,
    #three_best_models_combined_2000_df,
    four_best_models_combined_2000_df,
    all_combined_2000_df,
]).reset_index()
results_combined_2000_df

In [ ]:
results_combined_500_each_df = pd.concat([
    results_500_each_df,
    combined_hammer_500_each_df,
    combined_hammer_coqgym_500_each_df,
    combined_hammer_no_rescale_500_each_df,
    combined_hammer_no_rescale_coqgym_500_each_df,
    two_best_models_combined_500_each_df,
]).reset_index()
results_combined_500_each_df

In [ ]:
results_combined_500_each_df["run"].unique()

## Simple statistics for the data

In [ ]:
results_2000_df.groupby("run")[["error", "omitted"]].agg(["sum"])

In [ ]:
results_500_each_df.groupby("run")["theorem"].agg(["count", "nunique"])

In [ ]:
# error rate
results_2000_df.groupby("run")["error"].agg(["sum", "mean"])

In [ ]:
# error rate
results_500_each_df.groupby("run")["error"].agg(["sum", "mean"])

In [ ]:
# error rate
df = results_500_each_df.groupby(["run", "package"])["error"].agg(["sum", "mean"])
df = df[df["mean"] > .05]
df

In [ ]:
results_500_each_df[results_500_each_df["run"].str.contains("k-NN")].groupby("package").size().sort_values()

In [ ]:
results_500_each_df.groupby(["run", "package"]).size()

In [ ]:
results_2000_df.groupby("run")["solved"].agg(["count", "sum", "mean"])

In [ ]:
df = results_2000_df.copy()
df["end_early"] = (results_2000_df["time"] < .95 * 600.0) & ~results_2000_df["solved"]
df.groupby("run")["end_early"].agg(["count", "sum", "mean"])

In [ ]:
df = results_500_each_df.copy()
df.pivot_table(values="solved", columns="run", index="package", aggfunc="sum")

In [ ]:
df = results_500_each_df.copy()
print(df.pivot_table(values="solved", columns="run", index="package", aggfunc="sum").to_latex())

In [ ]:
df = results_500_each_df.copy()
df.pivot_table(values="solved", columns="run", index="package", aggfunc="sum")[["GNN Names Update new definitions", "GNN No names Update new definitions", "k-NN"]]

In [ ]:
df = results_500_each_df.copy()
df = df.pivot_table(values="solved", columns="run", index="package", aggfunc="mean")
df

In [ ]:
df = results_500_each_df.copy()
print(df.pivot_table(values="solved", columns="run", index="package", aggfunc="mean").to_latex())

In [ ]:
x = """package total thms2000 thms500each sum
coq-bbv.1.3 653 59 441 500
coq-bits.1.1.0 428 35 393 428
coq-bytestring.0.9.0 19 1 18 19
coq-ceres.0.4.0 103 12 91 103
coq-corn.8.16.0 6757 584 0 584
coq-gaia-stern.1.15 923 62 438 500
coq-haskell.1.0.0 174 21 153 174
coq-hott.8.11 3883 342 158 500
coq-iris-heap-lang.3.4.0 331 38 293 331
coq-mathcomp-apery.1.0.1 512 38 462 500
coq-mathcomp-odd-order.1.14.0 1602 129 371 500
coq-poltac.0.8.11 309 20 289 309
coq-printf.2.0.0 21 1 20 21
coq-qcert.2.2.0 4565 341 159 500
coq-smtcoq.2.0+8.11 999 90 410 500
coq-tlc.20200328 2191 195 305 500
coq-topology.10.0.1 352 32 320 352"""
headers = x.split("\n")[0].split(" ")
data = [y.split(" ") for y in x.split("\n")[1:]]
package_sizes = pd.DataFrame(data, columns=headers)
package_sizes["total"] = package_sizes["total"].astype("int")
package_sizes

## Statistics for paper

In [ ]:
runs = {
    "GNN No names Update new definitions" : "G2T-Anon-Update",
    "GNN Names Update new definitions": "G2T-Named-Update",
    "GNN No Definitions Update no definitions": "G2T-NoDefs-Frozen",
    "Transformer GPU": "Transformer (GPU)",
    "CoqHammer combined": "CoqHammer combined",
    "CoqHammer combined (unscaled)": "CoqHammer combined (unscaled)",
    "CoqHammer combined on CoqGym": "CoqHammer combined",
    "CoqHammer combined (unscaled) on CoqGym": "CoqHammer combined (unscaled)",
    "k-NN on CoqGym": "k-NN",
}

time_limits = {
    "GNN No names Update new definitions" : 5,
    "GNN Names Update new definitions": 5,
    "GNN No Definitions Update no definitions": 5,
    "Transformer GPU": 5,
    "CoqHammer combined": 5,
    "CoqHammer combined": 5,
    "CoqHammer combined (unscaled)": 5,
    "CoqHammer combined on CoqGym": 10,
    "CoqHammer combined (unscaled) on CoqGym": 10,
    "k-NN on CoqGym": 10,
}

df = pd.concat([results_combined_2000_df.copy().assign(time_limit_min=10), results_combined_500_each_df.copy().assign(time_limit_min=5)])
display(df["run"].unique())
df = df[df["run"].isin(list(runs.keys()))]
display(df["run"].unique())
df = df[df["run"].map(time_limits) == df["time_limit_min"]]
df["run"] = df["run"].map(runs)
display(df["run"].unique())
df["package"] = df["package"].str.extract(r"coq-(.*)\.\d*\.\d*\.\d*")
display(df["package"].unique())
display(df.columns)
df = df.groupby(["package", "run", "time_limit_min"])["solved"].agg(["count", "sum"])
df = df.rename(columns={"count": "total", "sum": "solved"})
df = df.reset_index()
#df = df.melt(id_vars=["package", "run", "time_limit_min"], value_vars=["total", "solved"])
#df = df.pivot_table(index=["run", "time_limit_min"], values="value", columns=["package", "variable"])
df["category"] = "Ours"
package_results_ours_df = df
package_results_ours_df


In [ ]:
from io import StringIO

# data provided by Alex Sanchez-Stern
coqgym_results_tsv = """project	us	asTactic	tactok	tactok_asTac	hammer	as_hammer	total	coqgymTotal
angles	6	4	4	4	15	15	93	62
buchberger	178	70	76	80	166	192	750	725
chinese	58	31	35	40	56	58	137	131
dblib	73	41	45	52	55	67	192	180
dep-map	18	9	11	11	14	16	90	43
disel	258	83	89	107	185	194	824	634
hoare-tut	7	1	5	5	6	6	25	18
huffman	71	25	28	33	74	81	316	314
weak-up-to	24	23	18	25	30	36	150	139
zchinese	11	9	5	11	12	13	39	43
jordan-curve-theorem	89	19	22	24	165	168	663	628
zfc	56	33	33	38	64	70	241	237
tree-automata	242	96	111	126	292	311	817	828
coquelicot	187	95	100	109	273	299	1743	1467
fermat4	26	13	10	15	47	47	130	130
demos	60	50	53	54	54	55	69	68
coqoban	0	0	0	0	0	0	3	2
goedel	122	53	67	69	120	128	766	606
verdi-raft	270	117	121	136	337	351	2272	2127
verdi	128	37	47	53	122	127	618	514
zorns-lemma	18	10	12	13	18	21	156	149
fundamental-arithmetics	39	11	15	16	37	41	152	142
UnifySL	278	189	180	228	303	367	1149	968
coq-procrastination	20	5	6	6	3	5	25	8
PolTac	178	118	112	122	289	308	309	363"""

coqgym_results_io = StringIO(coqgym_results_tsv)
coq_gym_df = pd.read_csv(coqgym_results_io, sep="\t")

coq_gym_df["project"] = coq_gym_df["project"].str.lower()
packages = results_combined_2000_df["package"].str.extract(r"coq-(.*)\.\d*\.\d*\.\d*")[0]
coq_gym_df = coq_gym_df[coq_gym_df["project"].isin(packages)]


solvers = [
    ("us", "total", "ProverBot9001", 0, "ProverBot9001"),
    ("asTactic", "coqgymTotal", "ASTactic", 10, "CoqGym"),
    ("tactok", "coqgymTotal", "TacTok", 10, "CoqGym"),
    #("tactok_asTac", "coqgymTotal", "TacTok + ASTactic (unscaled)", 10),
    ("hammer", "coqgymTotal", "CoqHammer", 10, "CoqGym"),
    #("as_hammer", "coqgymTotal", "ASTactic + CoqHammer (unscaled)", 10),
]
dfs = []
for solved_col, total_col, solver_name, time_limit_min, category in solvers:
    df = coq_gym_df[["project", solved_col, total_col]]
    df = df.rename(columns={"project": "package", solved_col: "solved", total_col: "total"})
    df["run"] = solver_name
    df["time_limit_min"] = time_limit_min
    df["category"] = category
    dfs.append(df)
coq_gym_df = pd.concat(dfs)
coq_gym_df

In [ ]:
with open("emily_data/passport.json") as f:
    j = json.load(f)

projects_solved = {}
projects_total = {}
max_time = -1
for d in j["results"]:
    project = d["filename"].split("/")[2]
    if project in projects_total:
        projects_total[project] += 1
    else:
        projects_total[project] = 1
        projects_solved[project] = 0
    
    if d["time"] != "timeout":
        t = float(d["time"])
        if t > max_time:
            max_time = t
        projects_solved[project] += 1
projects_ratio = {p:projects_solved[p] / projects_total[p] for p in projects_total}
display(max_time, max_time / 60)
display(projects_ratio)
display(projects_solved)
display(projects_total)    

In [ ]:
from collections import defaultdict

with open("emily_data/passport.json") as f:
    j = json.load(f)

data = defaultdict(lambda: defaultdict(int))

max_time = -1
for d in j["results"]:
    project = d["filename"].split("/")[2]
    data[project]["total"] += 1
    
    if d["time"] != "timeout":
        data[project]["solved"] += 1

new_data = []
for p in data:
    new_data.append({
        "package": p.lower(),
        "solved": data[p]["solved"],
        "total": data[p]["total"],
        "run": "TacTok+Passport",
        "time_limit_min": 10,
        "category": "CoqGym"
    })
passport_df = pd.DataFrame(new_data)
packages = results_combined_2000_df["package"].str.extract(r"coq-(.*)\.\d*\.\d*\.\d*")[0]
passport_df = passport_df[passport_df["package"].isin(packages)]
passport_df
    

In [ ]:
from collections import defaultdict

with open("emily_data/diva.json") as f:
    j = json.load(f)

data = defaultdict(lambda: defaultdict(int))

max_time = -1
for d in j["results"]:
    project = d["filename"].split("/")[2]
    data[project]["total"] += 1
    
    if d["time"] != "timeout":
        data[project]["solved"] += 1

new_data = []
for p in data:
    new_data.append({
        "package": p.lower(),
        "solved": data[p]["solved"],
        "total": data[p]["total"],
        "run": "Diva",
        "time_limit_min": 10,
        "category": "CoqGym"
    })
diva_df = pd.DataFrame(new_data)
packages = results_combined_2000_df["package"].str.extract(r"coq-(.*)\.\d*\.\d*\.\d*")[0]
diva_df = diva_df[diva_df["package"].isin(packages)]
diva_df
    

In [ ]:
df = pd.concat([package_results_ours_df, coq_gym_df, diva_df, passport_df])
df["pass rate"] = df["solved"] / df["total"]
df["pass rate (solved/total)"] = df.apply(lambda row: f"{row['pass rate']:0.2f} ({row['solved']}/{row['total']})", axis=1)
display(df)
df = df.melt(id_vars=["category", "package", "run", "time_limit_min"], value_vars=["total", "solved", "pass rate"])
df = df.pivot_table(index=["category", "run", "time_limit_min"], values="value", columns=["package", "variable"])

for c in df.columns:
    print(c)
    if c[1] != "pass rate":
        print(df[c])
        df[c] = df[c].astype(int)
df = df.reset_index()
coqgym_comparison_df = df.copy()
coqgym_comparison_df

In [ ]:
df = pd.concat([package_results_ours_df, coq_gym_df, diva_df, passport_df])

# remove goedel since it has a lot of issues
df = df[df["package"] != "goedel"]

# add totals
df_ = df.groupby(["category", "run", "time_limit_min"])[["total", "solved"]].sum().reset_index()
df_["package"] = "ztotal"  # use z to put at end
df = pd.concat([df, df_])

df["pass rate"] = df["solved"] / df["total"]
df["pass rate (solved/total)"] = df.apply(lambda row: f"{row['pass rate']:0.3f} \\hfill ({row['solved']}/{row['total']})" if row["solved"] else "NA", axis=1)
df["time_limit_min"] = df["time_limit_min"].apply(lambda i: str(i) if i else "NA")

df = df.melt(id_vars=["category", "package", "run", "time_limit_min"], value_vars=["pass rate (solved/total)"])
df = df.rename(columns={"time_limit_min": "time (mins)"})
df = df.pivot_table(index=["category", "run", "time (mins)"], values="value", columns=["variable", "package"], aggfunc="first")
df.columns.names = ["", ""]  # get rid of annoying stuff
df = df.reset_index()

df = df.rename(columns={"ztotal": "total"})

coqgym_comparison_df = df.copy()
coqgym_comparison_df

In [ ]:
# all but poltac
df = coqgym_comparison_df.copy()
df = df.drop(('pass rate (solved/total)', 'poltac'), axis=1)
df = df.rename(columns={"run": ""})
for c in df.columns:
    if isinstance(c, tuple) and c[1]:
        df[c] = np.where(df[c].str.contains("NA"), np.nan, df[c])
df = df.dropna()
print(df.set_index("category").to_latex(
    index=False,
))

In [ ]:
# poltac
df = coqgym_comparison_df.copy()
df = df.rename(columns={"run": ""})
for c in df.columns:
    if isinstance(c, tuple) and c[1] and c[1] != "poltac":
        df = df.drop(c, axis=1)
print(df.set_index("category").to_latex(
    index=False,
))

Make two tables by copy and pasting the relevant lines from above into these tables.
Mark the best as bold with `\textbf{}`.

```latex
\begin{tabular}{lr|ccccc|c}
\toprule
& time & \multicolumn{6}{c}{pass rate (solved/total)} \\
& (min) & buchberger & coquelicot & hoare-tut & huffman & zorns-lemma & total\\
\midrule
ASTactic & 10 & 0.097 \hfill (70/725) & 0.065 \hfill (95/1467) & 0.056 \hfill (1/18) & 0.080 \hfill (25/314) & 0.067 \hfill (10/149) & 0.105 \hfill (319/3036) \\
CoqHammer & 10 & 0.229 \hfill (166/725) & 0.186 \hfill (273/1467) & \textbf{0.333} \hfill (6/18) & \textbf{0.236} \hfill (74/314) & 0.121 \hfill (18/149) & 0.272 \hfill (826/3036) \\
TacTok & 10 & 0.105 \hfill (76/725) & 0.068 \hfill (100/1467) & 0.278 \hfill (5/18) & 0.089 \hfill (28/314) & 0.081 \hfill (12/149) & 0.110 \hfill (333/3036) \\
\midrule
ProverBot9001 & NA & 0.237 \hfill (178/750) & 0.107 \hfill (187/1743) & 0.280 \hfill (7/25) & 0.225 \hfill (71/316) & 0.115 \hfill (18/156) & 0.194 \hfill (639/3299) \\
\midrule
CH combined & 10 & \textbf{0.284} \hfill (213/750) & 0.148 \hfill (258/1743) & 0.320 \hfill (8/25) & 0.207 \hfill (65/314) & 0.100 \hfill (26/259) & 0.242 \hfill (822/3400) \\
k-NN & 10 & 0.264 \hfill (198/750) & \textbf{0.234} \hfill (408/1743) & 0.320 \hfill (8/25) & 0.169 \hfill (53/314) & \textbf{0.282} \hfill (73/259) & \textbf{0.284} \hfill (967/3400) \\
\bottomrule
\end{tabular}
```

```latex
\begin{tabular}{lr|c}
\toprule
& time & \multicolumn{1}{c}{pass rate (solved/total)} \\
& (min) & poltac \\
\midrule
ASTactic & 10 & 0.325 \hfill (118/363) \\
CoqHammer & 10 & 0.796 \hfill (289/363) \\
TacTok & 10 & 0.309 \hfill (112/363) \\
\midrule
ProverBot9001 & NA & 0.576 \hfill (178/309) \\
\midrule
CoqHammer combined & 10 & 0.816 \hfill (252/309) \\
G2T-Anon-Update & 5 & 0.864 \hfill (267/309) \\
G2T-Named-Update & 5 & \textbf{0.867} \hfill (268/309) \\
G2T-NoDefs-Frozen & 5 & 0.773 \hfill (239/309) \\
Transformer (GPU) & 5 & 0.560 \hfill (173/309) \\
k-NN & 10 & 0.735 \hfill (227/309) \\
\bottomrule
\end{tabular}
```

OLD Table:

```latex
\begin{tabular}{lr|ccccc|c}
\toprule
& time & \multicolumn{6}{c}{pass rate (solved/total)} \\
& (min) & buchberger & coquelicot & hoare-tut & huffman & zorns-lemma & total\\
\midrule
ASTactic & 10 & 0.097 \hfill (70/725) & 0.065 \hfill (95/1467) & 0.056 \hfill (1/18) & 0.080 \hfill (25/314) & 0.067 \hfill (10/149) & 0.105 \hfill (319/3036) \\
CoqHammer & 10 & 0.229 \hfill (166/725) & 0.186 \hfill (273/1467) & \textbf{0.333} \hfill (6/18) & \textbf{0.236} \hfill (74/314) & 0.121 \hfill (18/149) & 0.272 \hfill (826/3036) \\
TacTok & 10 & 0.105 \hfill (76/725) & 0.068 \hfill (100/1467) & 0.278 \hfill (5/18) & 0.089 \hfill (28/314) & 0.081 \hfill (12/149) & 0.110 \hfill (333/3036) \\
\midrule
ProverBot9001 & NA & 0.237 \hfill (178/750) & 0.107 \hfill (187/1743) & 0.280 \hfill (7/25) & 0.225 \hfill (71/316) & 0.115 \hfill (18/156) & 0.194 \hfill (639/3299) \\
\midrule
k-NN & 10 & \textbf{0.264} \hfill (198/750) & \textbf{0.234} \hfill (408/1743) & 0.320 \hfill (8/25) & 0.169 \hfill (53/314) & \textbf{0.282} \hfill (73/259) & \textbf{0.284} \hfill (967/3400) \\
\bottomrule
\end{tabular}
```

```latex
\begin{tabular}{lr|c}
\toprule
& time & \multicolumn{1}{c}{pass rate (solved/total)} \\
& (min) & poltac \\
\midrule
ASTactic & 10 & 0.325 \hfill (118/363) \\
CoqHammer & 10 & 0.796 \hfill (289/363) \\
TacTok & 10 & 0.309 \hfill (112/363) \\
\midrule
ProverBot9001 & NA & 0.576 \hfill (178/309) \\
\midrule
CoqHammer combined & 5 & 0.780 \hfill (241/309) \\
G2T-Anon-Update & 5 & 0.864 \hfill (267/309) \\
G2T-Named-Update & 5 & \textbf{0.867} \hfill (268/309) \\
G2T-NoDefs-Frozen & 5 & 0.773 \hfill (239/309) \\
Transformer (GPU) & 5 & 0.560 \hfill (173/309) \\
k-NN & 10 & 0.735 \hfill (227/309) \\
\bottomrule
\end{tabular}
```

## Double check coqhammer data

In [ ]:
df = results_combined_500_each_df.copy()
df = df[df["run"].str.contains("CoqHammer")]
df.pivot_table(values="solved", columns="package", index="run", aggfunc="sum")

In [ ]:
df = results_combined_2000_df.copy()
df = df[df["run"].str.contains("CoqHammer")]
df.pivot_table(values="solved", columns="package", index="run", aggfunc="sum")